In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"


tf.config.list_physical_devices("GPU")


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [6]:
import tensorflow as tf

tf.config.list_physical_devices("GPU")
import warnings
warnings.filterwarnings('ignore')
from contextlib import redirect_stdout, redirect_stderr
from tensorflow import keras
import sys
import contextlib
tf.__version__

'2.10.0'

In [3]:
import gym
import numpy as np
import matplotlib.pyplot as plt

import gym
print(gym.__version__)
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from IPython import display

0.26.2


In [4]:
import tensorflow_probability as tfp


In [12]:
!nvidia-smi

Tue Oct 24 00:21:43 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.61.05    Driver Version: 520.61.05    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A6000    On   | 00000000:1A:00.0 Off |                  Off |
| 30%   35C    P2    67W / 300W |  47569MiB / 49140MiB |      1%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A6000    On   | 00000000:1B:00.0 Off |                  Off |
| 30%   

In [22]:
def preprocess(image):
    """Preprocess a frame from the game."""
    image = image[35:195]  # Crop
    image = image[::2, ::2, 0]  # Downsample by a factor of 2
    image[image == 144] = 0  # Erase background (background type 1)
    image[image == 109] = 0  # Erase background (background type 2)
    image[image != 0] = 1  # Set paddles and ball to 1, everything else to 0
    return np.reshape(image.astype(np.float32).ravel(), [80, 80])


# Neural Network 

In [23]:

# hyperparameters
learning_rate = 0.0001
gamma = 0.99

env = gym.make("Pong-v0")

A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]


In [24]:
env.reset()[0].shape

(210, 160, 3)

In [25]:

model_policy = tf.keras.Sequential()
model_policy.add(tf.keras.Input(shape=(80, 80, 2)))

model_policy.add(tf.keras.layers.Flatten())
model_policy.add(tf.keras.layers.Dense(256, activation='linear'))
#model.add(tf.keras.layers.Dense(256, activation='linear'))
model_policy.add(tf.keras.layers.Dense(2, activation='softmax'))
optimizer_policy = tf.keras.optimizers.Adam(learning_rate)


2023-10-23 23:45:54.412045: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-23 23:45:59.355467: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9269 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:1b:00.0, compute capability: 7.5


In [26]:
'''
class Value(tf.keras.Model):
    def __init__(self):
        super(Value, self).__init__()
        self.l1 = keras.Input(shape=(80, 80, 2))
        self.l2 = keras.layers.Dense(12, activation='relu', input_shape=(32,))
        self.l3  = keras.layers.Dense(1, activation='linear')

    def forward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        
        return x
    
model_value  = Value()
'''
model_value = tf.keras.Sequential()
model_value.add(tf.keras.Input(shape=(80, 80, 2)))
model_value.add(tf.keras.layers.Flatten())
model_value.add(tf.keras.layers.Dense(32, activation='linear'))
model_value.add(tf.keras.layers.Dense(1, activation='linear'))

optimizer_value = tf.keras.optimizers.Adam(learning_rate)


## Load model weights from episode 1500 previously. 

In [13]:
from tensorflow.keras.models import load_model

model_policy = load_model('policy_model_1500')
#model_value = load_model('policy_model_' + str(batch))


2023-10-24 00:39:12.981935: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-24 00:39:16.160764: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 454 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:b1:00.0, compute capability: 8.6


## continuing the model

In [ ]:

# training loop

last_100_episodes =[]
episode_rewards = []
for batch in range (0, 10000):
    num_episodes = 1
# Training loop
    obs_history_batch = []
    reward_history_batch = []
    action_history_batch = []
    discounted_rewards_batch = []
    
    
    loss_policy = 0
    mse_loss = 0
    #with tf.GradientTape() as tape:
    with tf.GradientTape(persistent=True) as tape_policy:
        with tf.GradientTape(persistent=True) as tape_value:
        #tape.watch(model.trainable_variables)
    
            for episode in range(num_episodes):
                obs_history_eps = []
                reward_history_eps = []
                action_history_eps = []
                selected_probs_eps = []

                index_history_eps = []
                prob_output_eps = tf.Variable(tf.zeros([0, 2], dtype=tf.float32))
                Vt_eps = tf.Variable(tf.zeros([0, 1], dtype=tf.float32))


                state = preprocess(env.reset()[0])
                prev_state = state
                episode_reward = 0
                #reward_history = []
                action_probs = []
                #selected_probs_eps = []

                terminated = False
                truncated = False 

            #with tf.GradientTape() as tape:

                while (not terminated) and (not truncated):

                    # Predict the probability of each action
                    with open('output.log', 'w') as log_file:
                        with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                            if episode_reward==0:
                                combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(prev_state)], axis=-1)
                                action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]
                            else: 
                                action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]

                    dist = tfp.distributions.Categorical(probs=action_prob, dtype=tf.float32)
                    action = dist.sample()
                    #print(int(action))
                    if int(action)==0:
                        a=2
                    elif int(action)==1:
                        a=3

                    #print(a)
                    Vt = model_value(np.expand_dims(combined_states, axis=0))
                    #print(Vt)
                    Vt_eps = tf.concat([Vt_eps, Vt], axis=0)

                    index_history_eps.append(int(action))
                    prob_output_eps = tf.concat([prob_output_eps, tf.expand_dims(action_prob, 0)], axis=0)

                    #print(prob_output_eps)


                    action_probs.append(action_prob)
                    selected_probs_eps.append(action_prob[int(action)])
                    # Take the chosen action
                    next_state, reward, terminated, truncated, info = env.step(a)
                    reward_history_eps.append(reward)
                    action_history_eps.append(a)
                    reward_history_batch.append(reward)
                    action_history_batch.append(a)


                    #print(f"CartPole-v0, episode {episode + batch*20}, score {sum(reward_history_eps)}")

                    next_state = preprocess(next_state)
                    episode_reward += reward
                    #reward_history.append(reward)
                    combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(next_state)], axis=-1)
                    obs_history_eps.append(combined_states)

                    prev_state = next_state
                    #if terminated or truncated:
                        #episode_rewards.append(episode_reward)
                episode_rewards.append(episode_reward)
                print(f"Episode: {episode+batch*num_episodes}, Reward: {episode_reward}")
                if len(episode_rewards) >= 100:
                    avg_reward = np.mean(episode_rewards[-100:])
                    print(f"Average Reward (Last 100 Episodes): {avg_reward}")
                        #break
        #after the end of the episode   
                discounted_rewards_eps = []
                reward_sum = 0
                for r in reward_history_eps[::-1]:
                    reward_sum = r + gamma * reward_sum
                    discounted_rewards_eps.insert(0, reward_sum)



                #predicted_action_prob = model(np.expand_dims(combined_states, axis=0)) #shape of two
                #loss = -tf.reduce_sum(tf.math.log(predicted_action_prob[0][a]) *discounted_rewards)
                discounted_rewards_eps = np.array(discounted_rewards_eps)
                reshaped_discounted_rewards = discounted_rewards_eps.reshape(-1, 1)
                #tensor_logprob = tf.math.log(tf.convert_to_t(selected_probs_eps, dtype=tf.float32))


                #probs_tensor = tf.math.log(model(np.expand_dims(obs_history_eps, axis=0))[0])
                #selecting probs
                index_history_eps = tf.cast(index_history_eps, dtype=tf.int32)  # Or tf.int64, depending on your needs

                selected_values = tf.math.log(tf.gather(prob_output_eps, index_history_eps, axis=1))

                tensor_logprob = tf.math.log(tf.convert_to_tensor(selected_probs_eps, dtype=tf.float32))
                tensor_rewards = tf.convert_to_tensor(reshaped_discounted_rewards, dtype=tf.float32)
                ###normalize the discounted rewards
                mean = tf.reduce_mean(tensor_rewards)
                std = tf.math.reduce_std(tensor_rewards)

                # Normalize the discounted rewards
                normalized_rewards = (tensor_rewards - mean) / (std + 1e-8)
                tensor_rewards_normalized = tf.convert_to_tensor(normalized_rewards, dtype=tf.float32)

                #loss =loss +  tf.reduce_sum (index_history_eps * tensor_rewards_normalized)
                for idx in range (0, tensor_rewards_normalized.shape[0]):
                    loss_policy = loss_policy + (tensor_rewards_normalized[idx]-Vt_eps[idx]) * tf.math.log(prob_output_eps[idx][int(index_history_eps[idx])])


            ##After one batch 
            #print(Vt_eps)
            loss_policy = -tf.convert_to_tensor(loss_policy)
            mse_loss = tf.keras.losses.mean_squared_error(Vt_eps, tensor_rewards_normalized)

    grads_policy = tape_policy.gradient(loss_policy, model_policy.trainable_variables)
    optimizer_policy.apply_gradients(zip(grads_policy, model_policy.trainable_variables))

    # Calculate the simple moving average of the last 100 episode rewards
    grads_value = tape_value.gradient(mse_loss, model_value.trainable_variables)
    optimizer_value.apply_gradients(zip(grads_value, model_value.trainable_variables))
    
    
    if batch%100==0 and batch!=0:
        model_policy.save('policy_model_afterloadingmodel_'+str(batch))  # This will create a 'my_model' directory with model parameters
        model_value.save('value_model_afterloadingmodel_'+str(batch))  # This will create a 'my_model' directory with model parameters

        warnings.filterwarnings('ignore')

        episode_numbers = list(range(1, len(episode_rewards) + 1))
        plt.plot(episode_numbers, episode_rewards, label="Episode Reward")
        #moving_avg = np.convolve(episode_rewards, np.ones(100) / 100, mode='valid')
        #plt.plot(episode_numbers[-len(moving_avg):], moving_avg, label="Moving Average (100 episodes)")
        plt.xlabel("Episode")
        plt.ylabel("Reward")
        plt.legend()
        #plt.show()
        plt.savefig('episode_rewards_afterloadingmodel_'+ str(batch)+'.png')
        np_eps_rewards = np.array(episode_rewards)
        np.save('np_eps_rewards_afterloadingmodel_'+str(batch)+'.npy', np_eps_rewards)


env.close()



Episode: 0, Reward: -21.0


INFO:tensorflow:Assets written to: policy_model_0/assets


INFO:tensorflow:Assets written to: policy_model_0/assets


Episode: 1, Reward: -21.0
Episode: 2, Reward: -21.0
Episode: 3, Reward: -21.0
Episode: 4, Reward: -18.0
Episode: 5, Reward: -21.0
Episode: 6, Reward: -19.0
Episode: 7, Reward: -21.0
Episode: 8, Reward: -21.0
Episode: 9, Reward: -19.0
Episode: 10, Reward: -20.0
Episode: 11, Reward: -20.0
Episode: 12, Reward: -21.0
Episode: 13, Reward: -21.0
Episode: 14, Reward: -20.0
Episode: 15, Reward: -20.0
Episode: 16, Reward: -19.0
Episode: 17, Reward: -21.0
Episode: 18, Reward: -21.0
Episode: 19, Reward: -20.0
Episode: 20, Reward: -21.0
Episode: 21, Reward: -19.0
Episode: 22, Reward: -16.0
Episode: 23, Reward: -20.0
Episode: 24, Reward: -18.0
Episode: 25, Reward: -21.0
Episode: 26, Reward: -20.0
Episode: 27, Reward: -17.0
Episode: 28, Reward: -19.0
Episode: 29, Reward: -20.0
Episode: 30, Reward: -20.0
Episode: 31, Reward: -21.0
Episode: 32, Reward: -18.0
Episode: 33, Reward: -20.0
Episode: 34, Reward: -16.0
Episode: 35, Reward: -21.0
Episode: 36, Reward: -21.0
Episode: 37, Reward: -17.0
Episode: 3

INFO:tensorflow:Assets written to: policy_model_500/assets


...

INFO:tensorflow:Assets written to: policy_model_1000/assets


Episode: 1001, Reward: -15.0
Average Reward (Last 100 Episodes): -14.5
Episode: 1002, Reward: -14.0
Average Reward (Last 100 Episodes): -14.47
Episode: 1003, Reward: -11.0
Average Reward (Last 100 Episodes): -14.45
Episode: 1004, Reward: -11.0
Average Reward (Last 100 Episodes): -14.45
Episode: 1005, Reward: -17.0
Average Reward (Last 100 Episodes): -14.49
Episode: 1006, Reward: -10.0
Average Reward (Last 100 Episodes): -14.52
Episode: 1007, Reward: -17.0
Average Reward (Last 100 Episodes): -14.54
Episode: 1008, Reward: -5.0
Average Reward (Last 100 Episodes): -14.48
Episode: 1009, Reward: -13.0
Average Reward (Last 100 Episodes): -14.46
Episode: 1010, Reward: -10.0
Average Reward (Last 100 Episodes): -14.42
Episode: 1011, Reward: -12.0
Average Reward (Last 100 Episodes): -14.38
Episode: 1012, Reward: -17.0
Average Reward (Last 100 Episodes): -14.46
Episode: 1013, Reward: -20.0
Average Reward (Last 100 Episodes): -14.53
Episode: 1014, Reward: -12.0
Average Reward (Last 100 Episodes): -

INFO:tensorflow:Assets written to: policy_model_1500/assets


INFO:tensorflow:Assets written to: policy_model_1500/assets


Episode: 1501, Reward: -9.0
Average Reward (Last 100 Episodes): -13.14
Episode: 1502, Reward: -18.0
Average Reward (Last 100 Episodes): -13.19
Episode: 1503, Reward: -8.0
Average Reward (Last 100 Episodes): -13.1
Episode: 1504, Reward: -17.0
Average Reward (Last 100 Episodes): -13.08
Episode: 1505, Reward: -15.0
Average Reward (Last 100 Episodes): -13.06
Episode: 1506, Reward: -13.0
Average Reward (Last 100 Episodes): -13.01
Episode: 1507, Reward: -8.0
Average Reward (Last 100 Episodes): -12.92
Episode: 1508, Reward: -15.0
Average Reward (Last 100 Episodes): -12.89
Episode: 1509, Reward: -15.0
Average Reward (Last 100 Episodes): -12.91
Episode: 1510, Reward: -16.0
Average Reward (Last 100 Episodes): -12.93
Episode: 1511, Reward: -18.0
Average Reward (Last 100 Episodes): -12.98
Episode: 1512, Reward: -17.0
Average Reward (Last 100 Episodes): -12.98
Episode: 1513, Reward: -15.0
Average Reward (Last 100 Episodes): -12.97
Episode: 1514, Reward: -14.0
Average Reward (Last 100 Episodes): -12

In [1]:
episode_numbers = list(range(1, len(episode_rewards) + 1))
plt.plot(episode_numbers, episode_rewards, label="Episode Reward")
moving_avg = np.convolve(episode_rewards, np.ones(100) / 100, mode='valid')
plt.plot(episode_numbers[-len(moving_avg):], moving_avg, label="Moving Average (100 episodes)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.show()
plt.savefig('episode_rewards_'+ str(batch)+'.png')


NameError: name 'episode_rewards' is not defined

## Testing the Model

In [ ]:


env = gym.make("Pong-v0")

episode_totalrewards = []


for episode in range(500):
    #reward_history_eps = []
    terminated = False
    truncated = False
    episode_reward = 0
    state = preprocess(env.reset()[0])

    prev_state = state

    while (not terminated) and (not truncated):
        # sampling an action from the policy
        with open('output.log', 'w') as log_file:
            with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                if episode_reward==0:
                    combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(prev_state)], axis=-1)
                    action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]
                else:
                    action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]

        dist = tfp.distributions.Categorical(probs=action_prob, dtype=tf.float32)
        action = dist.sample()
        #print(int(action))
        if int(action)==0:
            a=2
        elif int(action)==1:
            a=3
        next_state, reward, terminated, truncated, info = env.step(a)
        next_state = preprocess(next_state)
        combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(next_state)], axis=-1)
        prev_state = next_state
        #reward_history_eps.append(reward)
        episode_reward = episode_reward+ reward

    episode_totalrewards.append(episode_reward)
    #print(f"Episode: {episode}, Reward: {episode_reward}")


# Close the environment
env.close()


In [ ]:
mean_reward = np.mean(episode_totalrewards)
std_reward = np.std(episode_totalrewards)


# Create a histogram
plt.hist(episode_totalrewards, bins=10, color='orange', alpha=0.7)
plt.xlabel('Episode Total Rewards')
plt.ylabel('Number of Episodes')
plt.title('Histogram of Episode Rewards')
plt.legend([f'Rewards (Mean: {mean_reward:.2f}, Std Dev: {std_reward:.2f})'])
plt.grid(True)




# Show the histogram
plt.show()
